In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install pennylane -q

print("✅ Setup complete!")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 100.3 MB/s eta 0:00:00
✅ Setup complete!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pennylane as qml
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

BASE_PATH = '/content/drive/MyDrive/HAM10000_Project'
IMG_DIR_1 = f'{BASE_PATH}/HAM10000_images_part_1'
IMG_DIR_2 = f'{BASE_PATH}/HAM10000_images_part_2'

print("✅ Imports complete!")

✅ Device: cpu
✅ Imports complete!


In [ ]:
# Load balanced metadata
balanced_metadata = pd.read_csv(f'{BASE_PATH}/balanced_metadata.csv')

# Add binary labels
malignant = ['mel', 'bcc', 'akiec']
balanced_metadata['binary_label'] = balanced_metadata['dx'].apply(lambda x: 1 if x in malignant else 0)

print("="*60)
print("📊 DATASET")
print("="*60)
print(f"Total: {len(balanced_metadata)} images")
print(f"\nClass distribution:")
print(balanced_metadata['dx'].value_counts().sort_index())
print("\n✅ Data loaded!")

📊 DATASET
Total: 3500 images

Class distribution:
dx
akiec    500
bcc      500
bkl      500
df       500
mel      500
nv       500
vasc     500
Name: count, dtype: int64

✅ Data loaded!


In [ ]:
class SkinDataset(Dataset):
    def __init__(self, metadata, img_dir_1, img_dir_2, transform=None):
        self.metadata = metadata.reset_index(drop=True)
        self.img_dir_1 = img_dir_1
        self.img_dir_2 = img_dir_2
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        img_id = row['original_id'] if 'original_id' in row and pd.notna(row['original_id']) else row['image_id']

        img_path = os.path.join(self.img_dir_1, f"{img_id}.jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(self.img_dir_2, f"{img_id}.jpg")

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        label = row.get('binary_label', row.get('label', 0))
        return image, label

print("✅ Dataset class defined!")

✅ Dataset class defined!


In [ ]:
# ResNet requires 224x224 images
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet normalization
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("✅ Transforms defined (224×224 for ResNet)!")

✅ Transforms defined (224×224 for ResNet)!


In [ ]:
# Split data
train_meta, test_meta = train_test_split(
    balanced_metadata,
    test_size=0.2,
    random_state=42,
    stratify=balanced_metadata['dx']
)

# Create datasets
train_dataset = SkinDataset(train_meta, IMG_DIR_1, IMG_DIR_2, transform_train)
test_dataset = SkinDataset(test_meta, IMG_DIR_1, IMG_DIR_2, transform_test)

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"✅ Binary loaders ready!")
print(f"Train: {len(train_meta)} samples ({len(train_loader)} batches)")
print(f"Test: {len(test_meta)} samples ({len(test_loader)} batches)")

✅ Binary loaders ready!
Train: 2800 samples (88 batches)
Test: 700 samples (22 batches)


In [ ]:
# 8-qubit quantum circuit
n_qubits = 8
dev = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev, interface='torch')
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    for i in range(n_qubits-1):
        qml.CNOT(wires=[i, i+1])
    for i in range(n_qubits):
        qml.RY(weights[i], wires=i)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

print("✅ Quantum circuit defined!")

✅ Quantum circuit defined!


In [ ]:
class ResNet50Binary(nn.Module):
    def __init__(self):
        super().__init__()
        # Load pre-trained ResNet50
        resnet = models.resnet50(weights='IMAGENET1K_V1')

        # Freeze early layers
        for param in list(resnet.parameters())[:-30]:
            param.requires_grad = False

        # Remove final layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])

        # Custom classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print("✅ ResNet50 Classical defined!")

✅ ResNet50 Classical defined!


In [ ]:
class ResNet50Quantum(nn.Module):
    def __init__(self):
        super().__init__()
        # Load pre-trained ResNet50
        resnet = models.resnet50(weights='IMAGENET1K_V1')

        # Freeze early layers
        for param in list(resnet.parameters())[:-30]:
            param.requires_grad = False

        # Remove final layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])

        # Reduce to 8 features for quantum
        self.fc_reduce = nn.Linear(2048, 128)
        self.fc_quantum = nn.Linear(128, 8)
        self.dropout = nn.Dropout(0.3)

        # Quantum parameters
        self.q_weights = nn.Parameter(torch.randn(8) * 0.01)

        # Output
        self.fc_out = nn.Linear(8, 2)

    def forward(self, x):
        # ResNet features
        x = self.features(x)
        x = x.view(x.size(0), -1)

        # Reduce dimensions
        x = torch.relu(self.fc_reduce(x))
        x = self.dropout(x)
        x = self.fc_quantum(x)

        # Quantum circuit
        q_out = []
        for sample in x:
            q_result = quantum_circuit(sample, self.q_weights)
            q_out.append(torch.stack(q_result).float())
        x = torch.stack(q_out)

        return self.fc_out(x)

print("✅ ResNet50 Quantum defined!")

✅ ResNet50 Quantum defined!


In [ ]:
def train_binary_model(model, train_loader, test_loader, model_name, num_epochs=15, lr=0.0001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

    best_acc = 0
    save_path = f'{BASE_PATH}/{model_name}_resnet50.pth'

    print(f"\n{'='*60}")
    print(f"🚀 Training {model_name}")
    print(f"{'='*60}\n")

    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 20 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")

        # Validation
        model.eval()
        preds, labels_list = [], []
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                outputs = model(images)
                preds.extend(torch.argmax(outputs, 1).cpu().numpy())
                labels_list.extend(labels.numpy())

        acc = accuracy_score(labels_list, preds)
        avg_loss = train_loss / len(train_loader)

        print(f"\n✅ Epoch {epoch+1}: Accuracy={acc*100:.1f}%, Loss={avg_loss:.4f}\n")

        scheduler.step(acc)

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), save_path)
            print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

    print(f"{'='*60}")
    print(f"🎉 Training Complete! Best: {best_acc*100:.1f}%")
    print(f"{'='*60}\n")

    return best_acc

print("✅ Training function ready!")

✅ Training function ready!


In [ ]:
print("🔵 Training Classical ResNet50 Binary...")
print("Expected time: ~15-20 mins on GPU\n")

resnet_classical = ResNet50Binary().to(device)
classical_binary_acc = train_binary_model(
    resnet_classical,
    train_loader,
    test_loader,
    'classical_binary',
    num_epochs=15,
    lr=0.0001
)

print(f"\n📊 Classical ResNet50 Binary: {classical_binary_acc*100:.1f}%")

🔵 Training Classical ResNet50 Binary...
Expected time: ~15-20 mins on GPU

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 120MB/s]



🚀 Training classical_binary

Epoch 1/15, Batch 0/88, Loss: 0.6160
Epoch 1/15, Batch 20/88, Loss: 0.7467
Epoch 1/15, Batch 40/88, Loss: 0.3177
Epoch 1/15, Batch 60/88, Loss: 0.4331
Epoch 1/15, Batch 80/88, Loss: 0.3834

✅ Epoch 1: Accuracy=83.3%, Loss=0.4853

💾 Saved! Best: 83.3%

Epoch 2/15, Batch 0/88, Loss: 0.4546
Epoch 2/15, Batch 20/88, Loss: 0.3873
Epoch 2/15, Batch 40/88, Loss: 0.2283
Epoch 2/15, Batch 60/88, Loss: 0.4210
Epoch 2/15, Batch 80/88, Loss: 0.5501

✅ Epoch 2: Accuracy=84.6%, Loss=0.3708

💾 Saved! Best: 84.6%

Epoch 3/15, Batch 0/88, Loss: 0.2195
Epoch 3/15, Batch 20/88, Loss: 0.3278
Epoch 3/15, Batch 40/88, Loss: 0.2867
Epoch 3/15, Batch 60/88, Loss: 0.2855
Epoch 3/15, Batch 80/88, Loss: 0.3975

✅ Epoch 3: Accuracy=84.7%, Loss=0.2880

💾 Saved! Best: 84.7%

Epoch 4/15, Batch 0/88, Loss: 0.2923
Epoch 4/15, Batch 20/88, Loss: 0.2243
Epoch 4/15, Batch 40/88, Loss: 0.1611
Epoch 4/15, Batch 60/88, Loss: 0.1994
Epoch 4/15, Batch 80/88, Loss: 0.1532

✅ Epoch 4: Accuracy=87.1

In [ ]:
print("🔬 Training Quantum ResNet50 Binary...")
print("Expected time: ~30-40 mins (quantum is slower)\n")

resnet_quantum = ResNet50Quantum().to(device)
quantum_binary_acc = train_binary_model(
    resnet_quantum,
    train_loader,
    test_loader,
    'quantum_binary',
    num_epochs=15,
    lr=0.00005  # Lower LR for quantum
)

print(f"\n📊 BINARY RESULTS:")
print(f"Classical ResNet50: {classical_binary_acc*100:.1f}%")
print(f"Quantum ResNet50:   {quantum_binary_acc*100:.1f}%")

🔬 Training Quantum ResNet50 Binary...
Expected time: ~30-40 mins (quantum is slower)


🚀 Training quantum_binary

Epoch 1/15, Batch 0/88, Loss: 0.7175
Epoch 1/15, Batch 20/88, Loss: 0.5840
Epoch 1/15, Batch 40/88, Loss: 0.5775
Epoch 1/15, Batch 60/88, Loss: 0.5534
Epoch 1/15, Batch 80/88, Loss: 0.5806

✅ Epoch 1: Accuracy=81.3%, Loss=0.5479

💾 Saved! Best: 81.3%

Epoch 2/15, Batch 0/88, Loss: 0.3605
Epoch 2/15, Batch 20/88, Loss: 0.3944
Epoch 2/15, Batch 40/88, Loss: 0.4259
Epoch 2/15, Batch 60/88, Loss: 0.4232
Epoch 2/15, Batch 80/88, Loss: 0.4425

✅ Epoch 2: Accuracy=84.4%, Loss=0.4346

💾 Saved! Best: 84.4%

Epoch 3/15, Batch 0/88, Loss: 0.4736
Epoch 3/15, Batch 20/88, Loss: 0.3585
Epoch 3/15, Batch 40/88, Loss: 0.3895
Epoch 3/15, Batch 60/88, Loss: 0.3755
Epoch 3/15, Batch 80/88, Loss: 0.3351

✅ Epoch 3: Accuracy=85.6%, Loss=0.3947

💾 Saved! Best: 85.6%

Epoch 4/15, Batch 0/88, Loss: 0.3657
Epoch 4/15, Batch 20/88, Loss: 0.3224
Epoch 4/15, Batch 40/88, Loss: 0.3913
Epoch 4/15, Batch

In [ ]:
# Malignant (3-class)
malignant_classes = ['mel', 'bcc', 'akiec']
malignant_df = balanced_metadata[balanced_metadata['dx'].isin(malignant_classes)].copy()
malignant_label_map = {'mel': 0, 'bcc': 1, 'akiec': 2}
malignant_df['label'] = malignant_df['dx'].map(malignant_label_map)

print(f"🔴 Malignant: {len(malignant_df)} samples")
print(malignant_df['dx'].value_counts())

🔴 Malignant: 1500 samples
dx
akiec    500
bcc      500
mel      500
Name: count, dtype: int64


In [ ]:
# Benign (4-class)
benign_classes = ['nv', 'bkl', 'vasc', 'df']
benign_df = balanced_metadata[balanced_metadata['dx'].isin(benign_classes)].copy()
benign_label_map = {'nv': 0, 'bkl': 1, 'vasc': 2, 'df': 3}
benign_df['label'] = benign_df['dx'].map(benign_label_map)

print(f"🟢 Benign: {len(benign_df)} samples")
print(benign_df['dx'].value_counts())

🟢 Benign: 2000 samples
dx
bkl     500
df      500
nv      500
vasc    500
Name: count, dtype: int64


In [ ]:
# Malignant loaders
mal_train, mal_test = train_test_split(malignant_df, test_size=0.2, random_state=42, stratify=malignant_df['dx'])
mal_train_ds = SkinDataset(mal_train, IMG_DIR_1, IMG_DIR_2, transform_train)
mal_test_ds = SkinDataset(mal_test, IMG_DIR_1, IMG_DIR_2, transform_test)
mal_train_loader = DataLoader(mal_train_ds, batch_size=32, shuffle=True)
mal_test_loader = DataLoader(mal_test_ds, batch_size=32, shuffle=False)

# Benign loaders
ben_train, ben_test = train_test_split(benign_df, test_size=0.2, random_state=42, stratify=benign_df['dx'])
ben_train_ds = SkinDataset(ben_train, IMG_DIR_1, IMG_DIR_2, transform_train)
ben_test_ds = SkinDataset(ben_test, IMG_DIR_1, IMG_DIR_2, transform_test)
ben_train_loader = DataLoader(ben_train_ds, batch_size=32, shuffle=True)
ben_test_loader = DataLoader(ben_test_ds, batch_size=32, shuffle=False)

print(f"✅ Sub-classifier loaders ready!")
print(f"Malignant: {len(mal_train)} train, {len(mal_test)} test")
print(f"Benign: {len(ben_train)} train, {len(ben_test)} test")

✅ Sub-classifier loaders ready!
Malignant: 1200 train, 300 test
Benign: 1600 train, 400 test


In [ ]:
class ResNet50SubClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Load pre-trained ResNet50
        resnet = models.resnet50(weights='IMAGENET1K_V1')

        # Freeze early layers
        for param in list(resnet.parameters())[:-30]:
            param.requires_grad = False

        # Remove final layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])

        # Custom classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print("✅ Classical Sub-Classifier defined!")

✅ Classical Sub-Classifier defined!


In [ ]:
class ResNet50QuantumSub(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Load pre-trained ResNet50
        resnet = models.resnet50(weights='IMAGENET1K_V1')

        # Freeze early layers
        for param in list(resnet.parameters())[:-30]:
            param.requires_grad = False

        # Remove final layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])

        # Reduce to 8 for quantum
        self.fc_reduce = nn.Linear(2048, 128)
        self.fc_quantum = nn.Linear(128, 8)
        self.dropout = nn.Dropout(0.4)

        # Quantum parameters
        self.q_weights = nn.Parameter(torch.randn(8) * 0.01)

        # Output
        self.fc_out = nn.Linear(8, num_classes)

    def forward(self, x):
        # ResNet features
        x = self.features(x)
        x = x.view(x.size(0), -1)

        # Reduce dimensions
        x = torch.relu(self.fc_reduce(x))
        x = self.dropout(x)
        x = self.fc_quantum(x)

        # Quantum circuit
        q_out = []
        for sample in x:
            q_result = quantum_circuit(sample, self.q_weights)
            q_out.append(torch.stack(q_result).float())
        x = torch.stack(q_out)

        return self.fc_out(x)

print("✅ Quantum Sub-Classifier defined!")

✅ Quantum Sub-Classifier defined!


In [ ]:
print("🔴 [1/4] Training Classical Malignant (FIXED)...")

# Calculate class weights
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(mal_train['label']), y=mal_train['label'])
class_weights = torch.FloatTensor(class_weights).to(device)
print(f"Class weights: {class_weights}")

# Retrain with weighted loss
classical_mal = ResNet50SubClassifier(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, classical_mal.parameters()), lr=0.00001)  # Much lower LR
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_acc = 0
save_path = f'{BASE_PATH}/classical_malignant_3class_resnet50.pth'

print(f"\n{'='*60}")
print(f"🚀 Training classical_malignant_3class (Fixed)")
print(f"{'='*60}\n")

for epoch in range(15):
    classical_mal.train()
    for batch_idx, (images, labels) in enumerate(mal_train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = classical_mal(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/15, Batch {batch_idx}/{len(mal_train_loader)}, Loss: {loss.item():.4f}")

    classical_mal.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in mal_test_loader:
            images = images.to(device)
            outputs = classical_mal(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}%\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(classical_mal.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

print(f"🎉 Best: {best_acc*100:.1f}%")
classical_mal_acc = best_acc

🔴 [1/4] Training Classical Malignant (FIXED)...
Class weights: tensor([1., 1., 1.], device='cuda:0')

🚀 Training classical_malignant_3class (Fixed)

Epoch 1/15, Batch 0/38, Loss: 1.0768
Epoch 1/15, Batch 10/38, Loss: 0.7154
Epoch 1/15, Batch 20/38, Loss: 0.4440
Epoch 1/15, Batch 30/38, Loss: 0.2898

✅ Epoch 1: 100.0%

💾 Saved! Best: 100.0%

Epoch 2/15, Batch 0/38, Loss: 0.1863
Epoch 2/15, Batch 10/38, Loss: 0.1446
Epoch 2/15, Batch 20/38, Loss: 0.0923
Epoch 2/15, Batch 30/38, Loss: 0.0700

✅ Epoch 2: 100.0%

Epoch 3/15, Batch 0/38, Loss: 0.0604
Epoch 3/15, Batch 10/38, Loss: 0.0441
Epoch 3/15, Batch 20/38, Loss: 0.0372
Epoch 3/15, Batch 30/38, Loss: 0.0288

✅ Epoch 3: 100.0%

Epoch 4/15, Batch 0/38, Loss: 0.0280
Epoch 4/15, Batch 10/38, Loss: 0.0262
Epoch 4/15, Batch 20/38, Loss: 0.0187
Epoch 4/15, Batch 30/38, Loss: 0.0184

✅ Epoch 4: 100.0%

Epoch 5/15, Batch 0/38, Loss: 0.0180
Epoch 5/15, Batch 10/38, Loss: 0.0136
Epoch 5/15, Batch 20/38, Loss: 0.0154
Epoch 5/15, Batch 30/38, Loss: 

In [ ]:
print("🟢 [2/4] Training Classical Benign Sub-Classifier...")
print("Expected time: ~20 mins\n")

classical_ben = ResNet50SubClassifier(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, classical_ben.parameters()), lr=0.00001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_acc = 0
save_path = f'{BASE_PATH}/classical_benign_4class_resnet50.pth'

print(f"\n{'='*60}")
print(f"🚀 Training classical_benign_4class")
print(f"{'='*60}\n")

for epoch in range(15):
    classical_ben.train()
    for batch_idx, (images, labels) in enumerate(ben_train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = classical_ben(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/15, Batch {batch_idx}/{len(ben_train_loader)}, Loss: {loss.item():.4f}")

    classical_ben.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in ben_test_loader:
            images = images.to(device)
            outputs = classical_ben(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}%\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(classical_ben.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

print(f"🎉 Classical Benign Best: {best_acc*100:.1f}%")
classical_ben_acc = best_acc

🟢 [2/4] Training Classical Benign Sub-Classifier...
Expected time: ~20 mins


🚀 Training classical_benign_4class

Epoch 1/15, Batch 0/50, Loss: 1.3294
Epoch 1/15, Batch 10/50, Loss: 0.9431
Epoch 1/15, Batch 20/50, Loss: 0.6719
Epoch 1/15, Batch 30/50, Loss: 0.4285
Epoch 1/15, Batch 40/50, Loss: 0.2847

✅ Epoch 1: 100.0%

💾 Saved! Best: 100.0%

Epoch 2/15, Batch 0/50, Loss: 0.1998
Epoch 2/15, Batch 10/50, Loss: 0.1512
Epoch 2/15, Batch 20/50, Loss: 0.1018
Epoch 2/15, Batch 30/50, Loss: 0.0772
Epoch 2/15, Batch 40/50, Loss: 0.0699

✅ Epoch 2: 100.0%

Epoch 3/15, Batch 0/50, Loss: 0.0500
Epoch 3/15, Batch 10/50, Loss: 0.0433
Epoch 3/15, Batch 20/50, Loss: 0.0347
Epoch 3/15, Batch 30/50, Loss: 0.0319
Epoch 3/15, Batch 40/50, Loss: 0.0296

✅ Epoch 3: 100.0%

Epoch 4/15, Batch 0/50, Loss: 0.0255
Epoch 4/15, Batch 10/50, Loss: 0.0231
Epoch 4/15, Batch 20/50, Loss: 0.0192
Epoch 4/15, Batch 30/50, Loss: 0.0192
Epoch 4/15, Batch 40/50, Loss: 0.0169

✅ Epoch 4: 100.0%

Epoch 5/15, Batch 0/50, Los

In [ ]:
print("🔬 [3/4] Training Quantum Malignant Sub-Classifier...")
print("Expected time: ~40-50 mins (quantum is slower)\n")

quantum_mal = ResNet50QuantumSub(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, quantum_mal.parameters()), lr=0.00001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_acc = 0
save_path = f'{BASE_PATH}/quantum_malignant_3class_resnet50.pth'

print(f"\n{'='*60}")
print(f"🚀 Training quantum_malignant_3class")
print(f"{'='*60}\n")

for epoch in range(15):
    quantum_mal.train()
    for batch_idx, (images, labels) in enumerate(mal_train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = quantum_mal(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/15, Batch {batch_idx}/{len(mal_train_loader)}, Loss: {loss.item():.4f}")

    quantum_mal.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in mal_test_loader:
            images = images.to(device)
            outputs = quantum_mal(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}%\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(quantum_mal.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

print(f"🎉 Quantum Malignant Best: {best_acc*100:.1f}%")
quantum_mal_acc = best_acc

🔬 [3/4] Training Quantum Malignant Sub-Classifier...
Expected time: ~40-50 mins (quantum is slower)


🚀 Training quantum_malignant_3class

Epoch 1/15, Batch 0/38, Loss: 1.6220
Epoch 1/15, Batch 10/38, Loss: 1.5517
Epoch 1/15, Batch 20/38, Loss: 1.4556
Epoch 1/15, Batch 30/38, Loss: 1.3583

✅ Epoch 1: 0.0%

Epoch 2/15, Batch 0/38, Loss: 1.2816
Epoch 2/15, Batch 10/38, Loss: 1.1879
Epoch 2/15, Batch 20/38, Loss: 1.1531
Epoch 2/15, Batch 30/38, Loss: 1.1171

✅ Epoch 2: 0.0%

Epoch 3/15, Batch 0/38, Loss: 1.0598
Epoch 3/15, Batch 10/38, Loss: 1.0276
Epoch 3/15, Batch 20/38, Loss: 0.9883
Epoch 3/15, Batch 30/38, Loss: 0.9401

✅ Epoch 3: 12.7%

💾 Saved! Best: 12.7%

Epoch 4/15, Batch 0/38, Loss: 0.9560
Epoch 4/15, Batch 10/38, Loss: 0.9027
Epoch 4/15, Batch 20/38, Loss: 0.8779
Epoch 4/15, Batch 30/38, Loss: 0.8829

✅ Epoch 4: 95.3%

💾 Saved! Best: 95.3%

Epoch 5/15, Batch 0/38, Loss: 0.8812
Epoch 5/15, Batch 10/38, Loss: 0.8641
Epoch 5/15, Batch 20/38, Loss: 0.8585
Epoch 5/15, Batch 30/38, L

KeyboardInterrupt: 

In [ ]:
print("🔬 [4/4] Training Quantum Benign Sub-Classifier...")
print("Expected time: ~40-50 mins\n")

quantum_ben = ResNet50QuantumSub(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, quantum_ben.parameters()), lr=0.00001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_acc = 0
save_path = f'{BASE_PATH}/quantum_benign_4class_resnet50.pth'

print(f"\n{'='*60}")
print(f"🚀 Training quantum_benign_4class")
print(f"{'='*60}\n")

for epoch in range(15):
    quantum_ben.train()
    for batch_idx, (images, labels) in enumerate(ben_train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = quantum_ben(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/15, Batch {batch_idx}/{len(ben_train_loader)}, Loss: {loss.item():.4f}")

    quantum_ben.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in ben_test_loader:
            images = images.to(device)
            outputs = quantum_ben(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}%\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(quantum_ben.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

print(f"🎉 Quantum Benign Best: {best_acc*100:.1f}%")
quantum_ben_acc = best_acc

🔬 [4/4] Training Quantum Benign Sub-Classifier...
Expected time: ~40-50 mins


🚀 Training quantum_benign_4class

Epoch 1/15, Batch 0/50, Loss: 1.1133
Epoch 1/15, Batch 10/50, Loss: 1.1013
Epoch 1/15, Batch 20/50, Loss: 1.0841
Epoch 1/15, Batch 30/50, Loss: 1.0484
Epoch 1/15, Batch 40/50, Loss: 1.0215

✅ Epoch 1: 100.0%

💾 Saved! Best: 100.0%

Epoch 2/15, Batch 0/50, Loss: 0.9987
Epoch 2/15, Batch 10/50, Loss: 0.9926
Epoch 2/15, Batch 20/50, Loss: 0.9861
Epoch 2/15, Batch 30/50, Loss: 0.9791
Epoch 2/15, Batch 40/50, Loss: 0.9733

✅ Epoch 2: 100.0%

Epoch 3/15, Batch 0/50, Loss: 0.9751
Epoch 3/15, Batch 10/50, Loss: 0.9602
Epoch 3/15, Batch 20/50, Loss: 0.9469
Epoch 3/15, Batch 30/50, Loss: 0.9387
Epoch 3/15, Batch 40/50, Loss: 0.9027

✅ Epoch 3: 100.0%

Epoch 4/15, Batch 0/50, Loss: 0.8834
Epoch 4/15, Batch 10/50, Loss: 0.8600
Epoch 4/15, Batch 20/50, Loss: 0.8634
Epoch 4/15, Batch 30/50, Loss: 0.8519
Epoch 4/15, Batch 40/50, Loss: 0.8432

✅ Epoch 4: 100.0%

Epoch 5/15, Batch 0/50, Loss

KeyboardInterrupt: 

In [ ]:
# Label mappings
malignant_labels = {0: 'mel', 1: 'bcc', 2: 'akiec'}
benign_labels = {0: 'nv', 1: 'bkl', 2: 'vasc', 3: 'df'}
all_classes = ['mel', 'bcc', 'akiec', 'nv', 'bkl', 'vasc', 'df']

# Use ORIGINAL full dataset for testing (not balanced)
original_metadata = pd.read_csv(f'{BASE_PATH}/HAM10000_metadata.csv')

# Pick 50 per class from original data (fresh, not used in training)
test_samples = []
for cls in all_classes:
    cls_data = original_metadata[original_metadata['dx'] == cls]
    sampled = cls_data.sample(n=50, random_state=99)  # random_state=99 (different from training split 42)
    test_samples.append(sampled)

test_df = pd.concat(test_samples).reset_index(drop=True)
print(f"📊 Test set: {len(test_df)} images (50 per class)")
print(test_df['dx'].value_counts().sort_index())

# Create test dataset
test_dataset_full = SkinDataset(test_df, IMG_DIR_1, IMG_DIR_2, transform_test)
test_loader_full = DataLoader(test_dataset_full, batch_size=1, shuffle=False)

print("✅ Fresh test set ready!")

📊 Test set: 350 images (50 per class)
dx
akiec    50
bcc      50
bkl      50
df       50
mel      50
nv       50
vasc     50
Name: count, dtype: int64
✅ Fresh test set ready!


In [ ]:
def hierarchical_predict(image, classical_bin, quantum_bin, classical_mal, classical_ben, quantum_mal, quantum_ben):
    results = {}

    image = image.unsqueeze(0).to(device)

    # Classical pipeline
    with torch.no_grad():
        # Binary
        classical_bin_out = classical_bin(image)
        classical_bin_pred = torch.argmax(classical_bin_out, 1).item()

        # Sub-classifier
        if classical_bin_pred == 1:  # Malignant
            sub_out = classical_mal(image)
            sub_pred = torch.argmax(sub_out, 1).item()
            results['classical'] = malignant_labels[sub_pred]
        else:  # Benign
            sub_out = classical_ben(image)
            sub_pred = torch.argmax(sub_out, 1).item()
            results['classical'] = benign_labels[sub_pred]

        results['classical_binary'] = classical_bin_pred

    # Quantum pipeline
    with torch.no_grad():
        # Binary
        quantum_bin_out = quantum_bin(image)
        quantum_bin_pred = torch.argmax(quantum_bin_out, 1).item()

        # Sub-classifier
        if quantum_bin_pred == 1:  # Malignant
            sub_out = quantum_mal(image)
            sub_pred = torch.argmax(sub_out, 1).item()
            results['quantum'] = malignant_labels[sub_pred]
        else:  # Benign
            sub_out = quantum_ben(image)
            sub_pred = torch.argmax(sub_out, 1).item()
            results['quantum'] = benign_labels[sub_pred]

        results['quantum_binary'] = quantum_bin_pred

    return results

# Run predictions on all 350 test images
classical_preds = []
quantum_preds = []
true_labels = []

print("🧪 Running predictions on 350 fresh images...")

for idx, (image, _) in enumerate(test_loader_full):
    image = image.squeeze(0)
    true_label = test_df.iloc[idx]['dx']

    result = hierarchical_predict(
        image,
        resnet_classical_bin, resnet_quantum_bin,
        classical_mal_model, classical_ben_model,
        quantum_mal_model, quantum_ben_model
    )

    classical_preds.append(result['classical'])
    quantum_preds.append(result['quantum'])
    true_labels.append(true_label)

    if idx % 50 == 0:
        print(f"  Processed {idx}/350...")

print(f"\n✅ Done! All 350 predictions complete!")

🧪 Running predictions on 350 fresh images...
  Processed 0/350...
  Processed 50/350...
  Processed 100/350...
  Processed 150/350...
  Processed 200/350...
  Processed 250/350...
  Processed 300/350...

✅ Done! All 350 predictions complete!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Calculate accuracies
classical_acc = accuracy_score(true_labels, classical_preds)
quantum_acc = accuracy_score(true_labels, quantum_preds)

print("=" * 60)
print("📊 FINAL HIERARCHICAL RESULTS (350 Fresh Images)")
print("=" * 60)

print(f"\n🔵 Classical Pipeline: {classical_acc*100:.1f}%")
print(f"🟣 Quantum Pipeline:  {quantum_acc*100:.1f}%")

print(f"\n{'='*60}")
print("📋 CLASSICAL - Per Class Results:")
print("=" * 60)
print(classification_report(true_labels, classical_preds, target_names=all_classes))

print(f"\n{'='*60}")
print("📋 QUANTUM - Per Class Results:")
print("=" * 60)
print(classification_report(true_labels, quantum_preds, target_names=all_classes))

# Per class comparison
print(f"\n{'='*60}")
print("⚔️  CLASS-WISE COMPARISON")
print("=" * 60)
print(f"{'Class':<10} {'Classical':<12} {'Quantum':<12} {'Winner'}")
print("-" * 45)
for cls in all_classes:
    cls_true = [1 if t == cls else 0 for t in true_labels]
    cls_classical = [1 if p == cls else 0 for p in classical_preds]
    cls_quantum = [1 if p == cls else 0 for p in quantum_preds]

    c_acc = accuracy_score(cls_true, cls_classical) * 100
    q_acc = accuracy_score(cls_true, cls_quantum) * 100
    winner = "🔵 Classical" if c_acc > q_acc else "🟣 Quantum" if q_acc > c_acc else "⚖️ Tie"
    print(f"{cls:<10} {c_acc:<12.1f} {q_acc:<12.1f} {winner}")

# Previous vs Now comparison
print(f"\n{'='*60}")
print("📈 IMPROVEMENT: PREVIOUS vs NOW")
print("=" * 60)
print(f"{'Pipeline':<20} {'Previous':<12} {'Now':<12} {'Improvement'}")
print("-" * 55)
print(f"{'Classical':<20} {'49.7%':<12} {classical_acc*100:<12.1f} {classical_acc*100 - 49.7:+.1f}%")
print(f"{'Quantum':<20} {'50.9%':<12} {quantum_acc*100:<12.1f} {quantum_acc*100 - 50.9:+.1f}%")

📊 FINAL HIERARCHICAL RESULTS (350 Fresh Images)

🔵 Classical Pipeline: 24.6%
🟣 Quantum Pipeline:  26.9%

📋 CLASSICAL - Per Class Results:
              precision    recall  f1-score   support

         mel       0.00      0.00      0.00        50
         bcc       0.26      0.98      0.42        50
       akiec       0.00      0.00      0.00        50
          nv       0.00      0.00      0.00        50
         bkl       0.00      0.00      0.00        50
        vasc       0.23      0.74      0.35        50
          df       0.00      0.00      0.00        50

    accuracy                           0.25       350
   macro avg       0.07      0.25      0.11       350
weighted avg       0.07      0.25      0.11       350


📋 QUANTUM - Per Class Results:
              precision    recall  f1-score   support

         mel       1.00      0.02      0.04        50
         bcc       0.31      1.00      0.47        50
       akiec       0.00      0.00      0.00        50
          nv    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
# Check what sub-classifiers are actually predicting
print("🔍 DIAGNOSTIC: Sub-classifier predictions\n")

# Test classical malignant on 10 malignant images
mal_images = test_df[test_df['dx'].isin(['mel','bcc','akiec'])].head(10)
print("🔴 Classical Malignant predictions:")
for idx in mal_images.index:
    img, _ = test_dataset_full[idx]
    img = img.unsqueeze(0).to(device)
    with torch.no_grad():
        out = classical_mal_model(img)
        pred = torch.argmax(out, 1).item()
        probs = torch.softmax(out, 1)
    print(f"  True: {mal_images.loc[idx,'dx']:<8} Pred: {malignant_labels[pred]:<8} Probs: {probs.cpu().numpy()}")

print("\n🟢 Classical Benign predictions:")
ben_images = test_df[test_df['dx'].isin(['nv','bkl','vasc','df'])].head(10)
for idx in ben_images.index:
    img, _ = test_dataset_full[idx]
    img = img.unsqueeze(0).to(device)
    with torch.no_grad():
        out = classical_ben_model(img)
        pred = torch.argmax(out, 1).item()
        probs = torch.softmax(out, 1)
    print(f"  True: {ben_images.loc[idx,'dx']:<8} Pred: {benign_labels[pred]:<8} Probs: {probs.cpu().numpy()}")

🔍 DIAGNOSTIC: Sub-classifier predictions

🔴 Classical Malignant predictions:
  True: mel      Pred: bcc      Probs: [[0.07898322 0.82869685 0.09231991]]
  True: mel      Pred: bcc      Probs: [[0.0894016 0.7776544 0.132944 ]]
  True: mel      Pred: bcc      Probs: [[0.05679007 0.846815   0.09639488]]
  True: mel      Pred: bcc      Probs: [[0.05858257 0.8710959  0.07032158]]
  True: mel      Pred: bcc      Probs: [[0.0822458  0.80451447 0.11323972]]
  True: mel      Pred: bcc      Probs: [[0.07112824 0.82699114 0.10188057]]
  True: mel      Pred: bcc      Probs: [[0.08215035 0.8254931  0.09235657]]
  True: mel      Pred: bcc      Probs: [[0.0764371  0.8286587  0.09490421]]
  True: mel      Pred: bcc      Probs: [[0.06193443 0.85913116 0.07893438]]
  True: mel      Pred: bcc      Probs: [[0.0748582  0.82992005 0.0952217 ]]

🟢 Classical Benign predictions:
  True: nv       Pred: nv       Probs: [[0.7633577  0.09190834 0.06281262 0.08192134]]
  True: nv       Pred: nv       Probs: [[0.797

In [ ]:
# Check what labels the sub-classifiers actually saw during training
print("🔍 Checking label distribution in training data\n")

print("Malignant df columns:", malignant_df.columns.tolist())
print("\nMalignant label column values:")
print(malignant_df['label'].value_counts())
print("\nMalignant dx column values:")
print(malignant_df['dx'].value_counts())

print("\n" + "="*40)

print("\nBenign df columns:", benign_df.columns.tolist())
print("\nBenign label column values:")
print(benign_df['label'].value_counts())
print("\nBenign dx column values:")
print(benign_df['dx'].value_counts())

# Also check what SkinDataset actually returns as label
print("\n🔍 What SkinDataset returns as label:")
mal_train_ds_check = SkinDataset(malignant_df.head(20), IMG_DIR_1, IMG_DIR_2, transform_test)
for i in range(5):
    img, label = mal_train_ds_check[i]
    true_dx = malignant_df.iloc[i]['dx']
    true_label = malignant_df.iloc[i]['label']
    print(f"  dx={true_dx}, label_col={true_label}, dataset_returns={label}")

🔍 Checking label distribution in training data

Malignant df columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'is_augmented', 'original_id', 'binary_label', 'label']

Malignant label column values:
label
2    500
1    500
0    500
Name: count, dtype: int64

Malignant dx column values:
dx
akiec    500
bcc      500
mel      500
Name: count, dtype: int64


Benign df columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'is_augmented', 'original_id', 'binary_label', 'label']

Benign label column values:
label
1    500
3    500
0    500
2    500
Name: count, dtype: int64

Benign dx column values:
dx
bkl     500
df      500
nv      500
vasc    500
Name: count, dtype: int64

🔍 What SkinDataset returns as label:
  dx=akiec, label_col=2, dataset_returns=1
  dx=akiec, label_col=2, dataset_returns=1
  dx=akiec, label_col=2, dataset_returns=1
  dx=akiec, label_col=2, dataset_returns=1
  dx=akiec, label_col=2, dataset_returns=1


In [ ]:
class SkinDatasetSub(Dataset):
    def __init__(self, metadata, img_dir_1, img_dir_2, transform=None):
        self.metadata = metadata.reset_index(drop=True)
        self.img_dir_1 = img_dir_1
        self.img_dir_2 = img_dir_2
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        img_id = row['original_id'] if 'original_id' in row and pd.notna(row['original_id']) else row['image_id']

        img_path = os.path.join(self.img_dir_1, f"{img_id}.jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(self.img_dir_2, f"{img_id}.jpg")

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # FIXED: Always use 'label' column, not binary_label
        label = int(row['label'])
        return image, label

# Verify fix
print("🔍 Verifying fix:")
mal_check = SkinDatasetSub(malignant_df.head(20), IMG_DIR_1, IMG_DIR_2, transform_test)
for i in range(6):
    img, label = mal_check[i]
    true_dx = malignant_df.iloc[i]['dx']
    true_label = malignant_df.iloc[i]['label']
    print(f"  dx={true_dx}, expected={true_label}, got={label} {'✅' if true_label == label else '❌'}")

🔍 Verifying fix:
  dx=akiec, expected=2, got=2 ✅
  dx=akiec, expected=2, got=2 ✅
  dx=akiec, expected=2, got=2 ✅
  dx=akiec, expected=2, got=2 ✅
  dx=akiec, expected=2, got=2 ✅
  dx=akiec, expected=2, got=2 ✅


In [ ]:
# Malignant loaders (fixed)
mal_train_ds = SkinDatasetSub(mal_train, IMG_DIR_1, IMG_DIR_2, transform_train)
mal_test_ds = SkinDatasetSub(mal_test, IMG_DIR_1, IMG_DIR_2, transform_test)
mal_train_loader = DataLoader(mal_train_ds, batch_size=32, shuffle=True)
mal_test_loader = DataLoader(mal_test_ds, batch_size=32, shuffle=False)

# Benign loaders (fixed)
ben_train_ds = SkinDatasetSub(ben_train, IMG_DIR_1, IMG_DIR_2, transform_train)
ben_test_ds = SkinDatasetSub(ben_test, IMG_DIR_1, IMG_DIR_2, transform_test)
ben_train_loader = DataLoader(ben_train_ds, batch_size=32, shuffle=True)
ben_test_loader = DataLoader(ben_test_ds, batch_size=32, shuffle=False)

# Quick verify labels are correct
print("🔍 Verifying loaders:")
for batch_imgs, batch_labels in mal_train_loader:
    print(f"Malignant batch labels: {batch_labels.unique().tolist()}")
    break

for batch_imgs, batch_labels in ben_train_loader:
    print(f"Benign batch labels: {batch_labels.unique().tolist()}")
    break

print("\n✅ Fixed loaders ready!")

🔍 Verifying loaders:
Malignant batch labels: [0, 1, 2]
Benign batch labels: [0, 1, 2, 3]

✅ Fixed loaders ready!


In [ ]:
print("🔴 [1/4] Retraining Classical Malignant (FIXED labels)...")

classical_mal_model = ResNet50SubClassifier(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, classical_mal_model.parameters()), lr=0.00001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_acc = 0
save_path = f'{BASE_PATH}/classical_malignant_3class_resnet50.pth'

for epoch in range(15):
    classical_mal_model.train()
    for batch_idx, (images, labels) in enumerate(mal_train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = classical_mal_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/15, Batch {batch_idx}/{len(mal_train_loader)}, Loss: {loss.item():.4f}")

    classical_mal_model.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in mal_test_loader:
            images = images.to(device)
            outputs = classical_mal_model(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    unique_preds = len(set(preds))
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}% | Predicting {unique_preds} classes\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(classical_mal_model.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

print(f"🎉 Classical Malignant Best: {best_acc*100:.1f}%")
classical_mal_acc = best_acc

🔴 [1/4] Retraining Classical Malignant (FIXED labels)...
Epoch 1/15, Batch 0/38, Loss: 1.1192
Epoch 1/15, Batch 10/38, Loss: 1.1123
Epoch 1/15, Batch 20/38, Loss: 1.1134
Epoch 1/15, Batch 30/38, Loss: 1.0250

✅ Epoch 1: 56.7% | Predicting 3 classes

💾 Saved! Best: 56.7%

Epoch 2/15, Batch 0/38, Loss: 1.0004
Epoch 2/15, Batch 10/38, Loss: 1.0116
Epoch 2/15, Batch 20/38, Loss: 0.9944
Epoch 2/15, Batch 30/38, Loss: 0.9409

✅ Epoch 2: 64.7% | Predicting 3 classes

💾 Saved! Best: 64.7%

Epoch 3/15, Batch 0/38, Loss: 0.9130
Epoch 3/15, Batch 10/38, Loss: 0.8653
Epoch 3/15, Batch 20/38, Loss: 0.8869
Epoch 3/15, Batch 30/38, Loss: 0.9399

✅ Epoch 3: 72.3% | Predicting 3 classes

💾 Saved! Best: 72.3%

Epoch 4/15, Batch 0/38, Loss: 0.8547
Epoch 4/15, Batch 10/38, Loss: 0.8083
Epoch 4/15, Batch 20/38, Loss: 0.7962
Epoch 4/15, Batch 30/38, Loss: 0.7880

✅ Epoch 4: 73.0% | Predicting 3 classes

💾 Saved! Best: 73.0%

Epoch 5/15, Batch 0/38, Loss: 0.7268
Epoch 5/15, Batch 10/38, Loss: 0.7988
Epoch 5/

In [ ]:
print("🟢 [2/4] Retraining Classical Benign (FIXED labels)...")

classical_ben_model = ResNet50SubClassifier(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, classical_ben_model.parameters()), lr=0.00001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_acc = 0
save_path = f'{BASE_PATH}/classical_benign_4class_resnet50.pth'

for epoch in range(15):
    classical_ben_model.train()
    for batch_idx, (images, labels) in enumerate(ben_train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = classical_ben_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/15, Batch {batch_idx}/{len(ben_train_loader)}, Loss: {loss.item():.4f}")

    classical_ben_model.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in ben_test_loader:
            images = images.to(device)
            outputs = classical_ben_model(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    unique_preds = len(set(preds))
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}% | Predicting {unique_preds} classes\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(classical_ben_model.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

print(f"🎉 Classical Benign Best: {best_acc*100:.1f}%")
classical_ben_acc = best_acc

🟢 [2/4] Retraining Classical Benign (FIXED labels)...
Epoch 1/15, Batch 0/50, Loss: 1.3569
Epoch 1/15, Batch 10/50, Loss: 1.3997
Epoch 1/15, Batch 20/50, Loss: 1.3470
Epoch 1/15, Batch 30/50, Loss: 1.3302
Epoch 1/15, Batch 40/50, Loss: 1.3680

✅ Epoch 1: 60.8% | Predicting 4 classes

💾 Saved! Best: 60.8%

Epoch 2/15, Batch 0/50, Loss: 1.3404
Epoch 2/15, Batch 10/50, Loss: 1.2754
Epoch 2/15, Batch 20/50, Loss: 1.2128
Epoch 2/15, Batch 30/50, Loss: 1.1923
Epoch 2/15, Batch 40/50, Loss: 1.1206

✅ Epoch 2: 71.0% | Predicting 4 classes

💾 Saved! Best: 71.0%

Epoch 3/15, Batch 0/50, Loss: 1.1110
Epoch 3/15, Batch 10/50, Loss: 1.1279
Epoch 3/15, Batch 20/50, Loss: 1.1344
Epoch 3/15, Batch 30/50, Loss: 1.0307
Epoch 3/15, Batch 40/50, Loss: 0.9787

✅ Epoch 3: 76.2% | Predicting 4 classes

💾 Saved! Best: 76.2%

Epoch 4/15, Batch 0/50, Loss: 0.8776
Epoch 4/15, Batch 10/50, Loss: 0.8155
Epoch 4/15, Batch 20/50, Loss: 0.7279
Epoch 4/15, Batch 30/50, Loss: 0.7861
Epoch 4/15, Batch 40/50, Loss: 0.708

In [ ]:
print("🔬 [3/4] Retraining Quantum Malignant (stronger settings)...")

quantum_mal_model = ResNet50QuantumSub(num_classes=3).to(device)

# Unfreeze more layers this time
for param in quantum_mal_model.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {'params': quantum_mal_model.features.parameters(), 'lr': 0.000005},  # Lower for ResNet
    {'params': quantum_mal_model.fc_reduce.parameters(), 'lr': 0.0001},   # Higher for new layers
    {'params': quantum_mal_model.fc_quantum.parameters(), 'lr': 0.0001},
    {'params': quantum_mal_model.q_weights, 'lr': 0.001},                 # Highest for quantum
    {'params': quantum_mal_model.fc_out.parameters(), 'lr': 0.0001}
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_acc = 0
save_path = f'{BASE_PATH}/quantum_malignant_3class_resnet50.pth'

for epoch in range(25):
    quantum_mal_model.train()
    for batch_idx, (images, labels) in enumerate(mal_train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = quantum_mal_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/25, Batch {batch_idx}/{len(mal_train_loader)}, Loss: {loss.item():.4f}")

    quantum_mal_model.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in mal_test_loader:
            images = images.to(device)
            outputs = quantum_mal_model(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    unique_preds = len(set(preds))
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}% | Predicting {unique_preds} classes\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(quantum_mal_model.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

    # Early stop if predicting all 3 classes above 75%
    if unique_preds == 3 and acc > 0.75:
        print(f"🎉 All 3 classes learned! Stopping early.")
        break

print(f"🎉 Quantum Malignant Best: {best_acc*100:.1f}%")
quantum_mal_acc = best_acc

🔬 [3/4] Retraining Quantum Malignant (stronger settings)...
Epoch 1/25, Batch 0/38, Loss: 1.2266
Epoch 1/25, Batch 10/38, Loss: 1.1265
Epoch 1/25, Batch 20/38, Loss: 1.0761
Epoch 1/25, Batch 30/38, Loss: 1.0425

✅ Epoch 1: 47.0% | Predicting 3 classes

💾 Saved! Best: 47.0%

Epoch 2/25, Batch 0/38, Loss: 1.0239
Epoch 2/25, Batch 10/38, Loss: 1.0537
Epoch 2/25, Batch 20/38, Loss: 1.0470
Epoch 2/25, Batch 30/38, Loss: 0.9783

✅ Epoch 2: 57.7% | Predicting 3 classes

💾 Saved! Best: 57.7%

Epoch 3/25, Batch 0/38, Loss: 0.9324
Epoch 3/25, Batch 10/38, Loss: 1.0276
Epoch 3/25, Batch 20/38, Loss: 0.9885
Epoch 3/25, Batch 30/38, Loss: 0.9105

✅ Epoch 3: 59.7% | Predicting 3 classes

💾 Saved! Best: 59.7%

Epoch 4/25, Batch 0/38, Loss: 0.9987
Epoch 4/25, Batch 10/38, Loss: 0.9247
Epoch 4/25, Batch 20/38, Loss: 1.0263
Epoch 4/25, Batch 30/38, Loss: 0.9044

✅ Epoch 4: 64.3% | Predicting 3 classes

💾 Saved! Best: 64.3%

Epoch 5/25, Batch 0/38, Loss: 0.9793
Epoch 5/25, Batch 10/38, Loss: 1.0066
Epoch

In [ ]:
print("🔬 Continuing Quantum Malignant training (no early stop)...")

# Model already trained to epoch 12, continue from there
for epoch in range(12, 25):
    quantum_mal_model.train()
    for batch_idx, (images, labels) in enumerate(mal_train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = quantum_mal_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/25, Batch {batch_idx}/{len(mal_train_loader)}, Loss: {loss.item():.4f}")

    quantum_mal_model.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in mal_test_loader:
            images = images.to(device)
            outputs = quantum_mal_model(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    unique_preds = len(set(preds))
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}% | Predicting {unique_preds} classes\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(quantum_mal_model.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

print(f"🎉 Quantum Malignant Final Best: {best_acc*100:.1f}%")
quantum_mal_acc = best_acc

🔬 Continuing Quantum Malignant training (no early stop)...
Epoch 13/25, Batch 0/38, Loss: 0.7070
Epoch 13/25, Batch 10/38, Loss: 0.8603
Epoch 13/25, Batch 20/38, Loss: 0.7190
Epoch 13/25, Batch 30/38, Loss: 0.7524

✅ Epoch 13: 78.0% | Predicting 3 classes

💾 Saved! Best: 78.0%

Epoch 14/25, Batch 0/38, Loss: 0.7135
Epoch 14/25, Batch 10/38, Loss: 0.7178
Epoch 14/25, Batch 20/38, Loss: 0.7580
Epoch 14/25, Batch 30/38, Loss: 0.7136

✅ Epoch 14: 76.3% | Predicting 3 classes

Epoch 15/25, Batch 0/38, Loss: 0.6919
Epoch 15/25, Batch 10/38, Loss: 0.7161
Epoch 15/25, Batch 20/38, Loss: 0.7384
Epoch 15/25, Batch 30/38, Loss: 0.8357

✅ Epoch 15: 78.3% | Predicting 3 classes

💾 Saved! Best: 78.3%

Epoch 16/25, Batch 0/38, Loss: 0.6924
Epoch 16/25, Batch 10/38, Loss: 0.6569
Epoch 16/25, Batch 20/38, Loss: 0.5958
Epoch 16/25, Batch 30/38, Loss: 0.8397

✅ Epoch 16: 81.3% | Predicting 3 classes

💾 Saved! Best: 81.3%

Epoch 17/25, Batch 0/38, Loss: 0.7024
Epoch 17/25, Batch 10/38, Loss: 0.7370
Epoch 

In [ ]:
print("🔬 [4/4] Training Quantum Benign (FIXED labels)...")

quantum_ben_model = ResNet50QuantumSub(num_classes=4).to(device)

# Unfreeze all layers
for param in quantum_ben_model.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {'params': quantum_ben_model.features.parameters(), 'lr': 0.000005},
    {'params': quantum_ben_model.fc_reduce.parameters(), 'lr': 0.0001},
    {'params': quantum_ben_model.fc_quantum.parameters(), 'lr': 0.0001},
    {'params': quantum_ben_model.q_weights, 'lr': 0.001},
    {'params': quantum_ben_model.fc_out.parameters(), 'lr': 0.0001}
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_acc = 0
save_path = f'{BASE_PATH}/quantum_benign_4class_resnet50.pth'

for epoch in range(25):
    quantum_ben_model.train()
    for batch_idx, (images, labels) in enumerate(ben_train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = quantum_ben_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/25, Batch {batch_idx}/{len(ben_train_loader)}, Loss: {loss.item():.4f}")

    quantum_ben_model.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in ben_test_loader:
            images = images.to(device)
            outputs = quantum_ben_model(images)
            preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            labels_list.extend(labels.numpy())

    acc = accuracy_score(labels_list, preds)
    unique_preds = len(set(preds))
    print(f"\n✅ Epoch {epoch+1}: {acc*100:.1f}% | Predicting {unique_preds} classes\n")

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(quantum_ben_model.state_dict(), save_path)
        print(f"💾 Saved! Best: {best_acc*100:.1f}%\n")

print(f"🎉 Quantum Benign Best: {best_acc*100:.1f}%")
quantum_ben_acc = best_acc

🔬 [4/4] Training Quantum Benign (FIXED labels)...
Epoch 1/25, Batch 0/50, Loss: 1.4707
Epoch 1/25, Batch 10/50, Loss: 1.4788
Epoch 1/25, Batch 20/50, Loss: 1.4133
Epoch 1/25, Batch 30/50, Loss: 1.4119
Epoch 1/25, Batch 40/50, Loss: 1.3712

✅ Epoch 1: 26.0% | Predicting 2 classes

💾 Saved! Best: 26.0%

Epoch 2/25, Batch 0/50, Loss: 1.3582
Epoch 2/25, Batch 10/50, Loss: 1.3941
Epoch 2/25, Batch 20/50, Loss: 1.3290
Epoch 2/25, Batch 30/50, Loss: 1.2962
Epoch 2/25, Batch 40/50, Loss: 1.3122

✅ Epoch 2: 42.5% | Predicting 3 classes

💾 Saved! Best: 42.5%

Epoch 3/25, Batch 0/50, Loss: 1.2776
Epoch 3/25, Batch 10/50, Loss: 1.1587
Epoch 3/25, Batch 20/50, Loss: 1.2079
Epoch 3/25, Batch 30/50, Loss: 1.2659
Epoch 3/25, Batch 40/50, Loss: 1.2710

✅ Epoch 3: 49.0% | Predicting 3 classes

💾 Saved! Best: 49.0%

Epoch 4/25, Batch 0/50, Loss: 1.3339
Epoch 4/25, Batch 10/50, Loss: 1.2573
Epoch 4/25, Batch 20/50, Loss: 1.2582
Epoch 4/25, Batch 30/50, Loss: 1.2104
Epoch 4/25, Batch 40/50, Loss: 1.3118

✅

In [ ]:
print("=" * 60)
print("🧪 FINAL END-TO-END HIERARCHICAL TEST (ResNet50)")
print("=" * 60)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Reload all 6 models with map_location
resnet_classical_bin = ResNet50Binary().to(device)
resnet_classical_bin.load_state_dict(torch.load(f'{BASE_PATH}/classical_binary_resnet50.pth', map_location=device))
resnet_classical_bin.eval()

resnet_quantum_bin = ResNet50Quantum().to(device)
resnet_quantum_bin.load_state_dict(torch.load(f'{BASE_PATH}/quantum_binary_resnet50.pth', map_location=device))
resnet_quantum_bin.eval()

classical_mal_model = ResNet50SubClassifier(num_classes=3).to(device)
classical_mal_model.load_state_dict(torch.load(f'{BASE_PATH}/classical_malignant_3class_resnet50.pth', map_location=device))
classical_mal_model.eval()

classical_ben_model = ResNet50SubClassifier(num_classes=4).to(device)
classical_ben_model.load_state_dict(torch.load(f'{BASE_PATH}/classical_benign_4class_resnet50.pth', map_location=device))
classical_ben_model.eval()

quantum_mal_model = ResNet50QuantumSub(num_classes=3).to(device)
quantum_mal_model.load_state_dict(torch.load(f'{BASE_PATH}/quantum_malignant_3class_resnet50.pth', map_location=device))
quantum_mal_model.eval()

quantum_ben_model = ResNet50QuantumSub(num_classes=4).to(device)
quantum_ben_model.load_state_dict(torch.load(f'{BASE_PATH}/quantum_benign_4class_resnet50.pth', map_location=device))
quantum_ben_model.eval()

print("✅ All 6 models reloaded!")

🧪 FINAL END-TO-END HIERARCHICAL TEST (ResNet50)
Device: cpu
✅ All 6 models reloaded!


In [ ]:
def hierarchical_predict(image, classical_bin, quantum_bin, classical_mal, classical_ben, quantum_mal, quantum_ben):
    results = {}

    image = image.unsqueeze(0).to(device)

    # Classical pipeline
    with torch.no_grad():
        # Binary
        classical_bin_out = classical_bin(image)
        classical_bin_pred = torch.argmax(classical_bin_out, 1).item()

        # Sub-classifier
        if classical_bin_pred == 1:  # Malignant
            sub_out = classical_mal(image)
            sub_pred = torch.argmax(sub_out, 1).item()
            results['classical'] = malignant_labels[sub_pred]
        else:  # Benign
            sub_out = classical_ben(image)
            sub_pred = torch.argmax(sub_out, 1).item()
            results['classical'] = benign_labels[sub_pred]

        results['classical_binary'] = classical_bin_pred

    # Quantum pipeline
    with torch.no_grad():
        # Binary
        quantum_bin_out = quantum_bin(image)
        quantum_bin_pred = torch.argmax(quantum_bin_out, 1).item()

        # Sub-classifier
        if quantum_bin_pred == 1:  # Malignant
            sub_out = quantum_mal(image)
            sub_pred = torch.argmax(sub_out, 1).item()
            results['quantum'] = malignant_labels[sub_pred]
        else:  # Benign
            sub_out = quantum_ben(image)
            sub_pred = torch.argmax(sub_out, 1).item()
            results['quantum'] = benign_labels[sub_pred]

        results['quantum_binary'] = quantum_bin_pred

    return results

print("✅ Hierarchical predict function defined!")

✅ Hierarchical predict function defined!


In [ ]:
# Run predictions on all 350 test images
classical_preds = []
quantum_preds = []
true_labels = []

print("🧪 Running hierarchical predictions on 350 fresh images...")
print("(This may take a few minutes on CPU)\n")

for idx, (image, _) in enumerate(test_loader_full):
    image = image.squeeze(0)
    true_label = test_df.iloc[idx]['dx']

    result = hierarchical_predict(
        image,
        resnet_classical_bin, resnet_quantum_bin,
        classical_mal_model, classical_ben_model,
        quantum_mal_model, quantum_ben_model
    )

    classical_preds.append(result['classical'])
    quantum_preds.append(result['quantum'])
    true_labels.append(true_label)

    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx+1}/350...")

print(f"\n✅ Done! All 350 predictions complete!")

🧪 Running hierarchical predictions on 350 fresh images...
(This may take a few minutes on CPU)

  Processed 50/350...
  Processed 100/350...
  Processed 150/350...
  Processed 200/350...
  Processed 250/350...
  Processed 300/350...
  Processed 350/350...

✅ Done! All 350 predictions complete!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Calculate accuracies
classical_acc = accuracy_score(true_labels, classical_preds)
quantum_acc = accuracy_score(true_labels, quantum_preds)

print("=" * 60)
print("📊 FINAL HIERARCHICAL RESULTS (350 Fresh Images)")
print("=" * 60)

print(f"\n🔵 Classical ResNet50 Pipeline: {classical_acc*100:.1f}%")
print(f"🟣 Quantum ResNet50 Pipeline:   {quantum_acc*100:.1f}%")

print(f"\n{'='*60}")
print("📋 CLASSICAL - Per Class Results:")
print("=" * 60)
print(classification_report(true_labels, classical_preds, target_names=all_classes))

print(f"\n{'='*60}")
print("📋 QUANTUM - Per Class Results:")
print("=" * 60)
print(classification_report(true_labels, quantum_preds, target_names=all_classes))

# Per class comparison
print(f"\n{'='*60}")
print("⚔️  CLASS-WISE COMPARISON")
print("=" * 60)
print(f"{'Class':<10} {'Classical':<12} {'Quantum':<12} {'Winner'}")
print("-" * 45)
for cls in all_classes:
    cls_true = [1 if t == cls else 0 for t in true_labels]
    cls_classical = [1 if p == cls else 0 for p in classical_preds]
    cls_quantum = [1 if p == cls else 0 for p in quantum_preds]

    c_acc = accuracy_score(cls_true, cls_classical) * 100
    q_acc = accuracy_score(cls_true, cls_quantum) * 100
    winner = "🔵 Classical" if c_acc > q_acc else "🟣 Quantum" if q_acc > c_acc else "⚖️ Tie"
    print(f"{cls:<10} {c_acc:<12.1f} {q_acc:<12.1f} {winner}")

# Improvement comparison
print(f"\n{'='*60}")
print("📈 IMPROVEMENT: Simple CNN vs ResNet50")
print("=" * 60)
print(f"{'Pipeline':<20} {'Simple CNN':<12} {'ResNet50':<12} {'Improvement'}")
print("-" * 55)
print(f"{'Classical':<20} {'49.7%':<12} {classical_acc*100:<12.1f} {classical_acc*100 - 49.7:+.1f}%")
print(f"{'Quantum':<20} {'50.9%':<12} {quantum_acc*100:<12.1f} {quantum_acc*100 - 50.9:+.1f}%")

print(f"\n{'='*60}")
print("🎉 SUMMARY")
print("=" * 60)
print(f"Binary Accuracy:  Classical {88.9}% | Quantum {90.4}% ✅")
print(f"Hierarchical Acc: Classical {classical_acc*100:.1f}% | Quantum {quantum_acc*100:.1f}%")
if quantum_acc > classical_acc:
    print(f"🏆 Quantum WINS by {(quantum_acc - classical_acc)*100:.1f}%!")
elif classical_acc > quantum_acc:
    print(f"🏆 Classical WINS by {(classical_acc - quantum_acc)*100:.1f}%!")
else:
    print(f"⚖️ TIE!")

📊 FINAL HIERARCHICAL RESULTS (350 Fresh Images)

🔵 Classical ResNet50 Pipeline: 78.9%
🟣 Quantum ResNet50 Pipeline:   80.6%

📋 CLASSICAL - Per Class Results:
              precision    recall  f1-score   support

         mel       0.71      0.82      0.76        50
         bcc       0.65      0.92      0.76        50
       akiec       0.76      0.58      0.66        50
          nv       0.93      0.84      0.88        50
         bkl       0.70      0.80      0.75        50
        vasc       0.94      0.60      0.73        50
          df       0.98      0.96      0.97        50

    accuracy                           0.79       350
   macro avg       0.81      0.79      0.79       350
weighted avg       0.81      0.79      0.79       350


📋 QUANTUM - Per Class Results:
              precision    recall  f1-score   support

         mel       0.74      0.62      0.67        50
         bcc       0.68      1.00      0.81        50
       akiec       0.89      0.62      0.73        